# Flujo de cálculo de retenciones con gastos proyectados

## Impuesto proyectado anual

Se calcula al inicio del año con base en:

`Ingreso bruto anual − Gastos proyectados = Base imponible estimada`

Sobre esa base se aplica la tabla progresiva → se obtiene el *impuesto proyectado anual*.

**División en 12 meses**

- El empleador divide ese impuesto anual entre 12.
- Ese valor es la *retención mensual* que se descuenta del sueldo.

**Sumatoria anual de retenciones**

- Al final del año, la suma de esas retenciones es el *total retenido*.
- Este valor se reporta en el Anexo de Relación de Dependencia (ARD) y aparece en la declaración.

## Comparación con impuesto real

En la declaración anual se recalcula el impuesto con los gastos reales comprobados.

Se compara:

`Impuesto real − Retenciones acumuladas = Saldo final`

- Si el resultado es positivo → se debe pagar.
- Si es negativo → se tiene saldo a favor (posible devolución).
- Si es cero → se cumplió exactamente la obligación.

**✅ En resumen:**

Proyección → retenciones mensuales (suavizan el pago durante el año).
Reales → impuesto definitivo (ajuste en la declaración).

*La diferencia entre ambos determina si se paga más, se recibe una devolución o se saldan las cuentas con el SRI.*

*Importante: se usó la normativa vigente en Ecuador para 2025. La fracción básica no gravada es de USD `12.081`, y el límite global de deducción de gastos personales es 1,3 veces esa fracción básica, es decir USD `15.705.3`. Aunque cada categoría de gasto tiene ese valor como tope individual, la suma de todas no puede superar el límite global de USD `15.705.3`.*

In [1]:
import importlib
import src.model

importlib.reload(src.model)  # Reloads the module to include recent modifications

<module 'src.model' from 'C:\\Users\\DESKTOP_A\\Desktop\\python\\code_projects\\ecu_ir_model\\src\\model.py'>

# Función de cálculo (inicio de año)

La función usa valores proyectados, tanto para gastos como para el ingreso.

In [1]:
from pathlib import Path
import pandas as pd
from src.model import InputsEnd, InputsStart, load_ir_table, load_expend_table, calculate, calculate_start

# Referencing the path with "__file__" is not supported in jupyter notebooks
PROJECT_ROOT = Path.cwd().parent.resolve()   # Sets the current working directory

# Paths to the required data
path_ir = PROJECT_ROOT / "data" / "tabla_ir.csv"
path_expend = PROJECT_ROOT / "data" / "gastos_proy.csv"

ir_table = load_ir_table(path_ir)
expend_table = load_expend_table(path_expend)

# Modify this value
income = 20000.0

inputs = InputsStart(
    income=income,
    projections=expend_table["proyeccion"].tolist(),
)

result_start = calculate_start(inputs, ir_table)
tb = pd.Series(result_start).to_frame(name="Valores")  # Neat visualization
print(tb)

                              Valores
Impuesto proyectado anual      145.95
Retención proyectada mensual    12.16


# Función de cálculo (Fin de año)

La función usa valores reales, junto a los valores iniciales de gastos proyectados.

In [2]:
# Paths to the required data
path_ir = PROJECT_ROOT / "data" / "tabla_ir.csv"
path_expend = PROJECT_ROOT / "data" / "gastos.csv"

# Load tables
ir_table = load_ir_table(path_ir)
expend_table = load_expend_table(path_expend)

# Modify the income and keep the projected retentions
income = 20000.0
retentions = result_start["Impuesto proyectado anual"]

inputs = InputsEnd(
    income=income,
    retentions=retentions,
    projections=expend_table["proyeccion"].tolist(),
    real_values=expend_table["real"].tolist()
)

result = calculate(inputs, ir_table)
tb = pd.Series(result).to_frame(name="Valores")
print(tb)

                            Valores
Base imponible proyectada  15000.00
Base imponible real        15600.00
Impuesto proyectado anual    145.95
Impuesto real anual          186.30
Retenciones acumuladas       145.95
Total a pagar                 40.35


# Caso de saldo a favor en el impuesto

Ocurre cuando el valor a pagar del impuesto tiene un saldo negativo. También es consecuencia de una base imponible real reducida. 

In [3]:
# Paths to the required data
path_ir = PROJECT_ROOT / "data" / "tabla_ir.csv"
path_expend = PROJECT_ROOT / "data" / "gastos_second_case.csv"

# Load tables
ir_table = load_ir_table(path_ir)
expend_table = load_expend_table(path_expend)

# Modify the income and keep the projected retentions
income = 20000.0
retentions = result_start["Impuesto proyectado anual"]

inputs = InputsEnd(
    income=income,
    retentions=retentions,
    projections=expend_table["proyeccion"].tolist(),
    real_values=expend_table["real"].tolist()
)

result = calculate(inputs, ir_table)
tb = pd.Series(result).to_frame(name="Valores")
print(tb)

                            Valores
Base imponible proyectada  15000.00
Base imponible real        12500.00
Impuesto proyectado anual    145.95
Impuesto real anual           20.95
Retenciones acumuladas       145.95
Total a pagar               -125.00


In [12]:
# Save the result as CSV if needed
path_sample = PROJECT_ROOT / "sample_result"

if not path_sample.exists():
    Path.mkdir(path_sample)
tb.to_csv(str(path_sample / "simulated_result.csv"), index=True)

# 📌 Nota sobre actualización de la tabla y límites de gastos deducibles

La tabla progresiva del Impuesto a la Renta y los límites de los gastos personales deducibles se actualizan periódicamente por el SRI en función de la inflación y las reformas tributarias vigentes.

**Tabla progresiva:** define los tramos de base imponible, el impuesto fijo de cada fracción básica y la tarifa aplicable al excedente. Una actualización puede cambiar los valores de fracción básica o las tarifas, afectando directamente el cálculo del impuesto causado.

**Gastos deducibles:** cada categoría (salud, educación, vivienda, alimentación, vestimenta) tiene un límite máximo anual y un límite global. Estos topes también pueden variar año a año. Superar los límites no genera mayor deducción, por lo que es fundamental conocerlos y aplicarlos correctamente.

**Importancia práctica:**

- Usar tablas o límites desactualizados puede producir cálculos erróneos de retenciones, impuestos proyectados y saldos finales.

- Para proyectos técnicos y simulaciones, mantener la tabla IR y los topes de gastos como archivos externos (CSV o parámetros) facilita su actualización sin modificar la lógica del código.

- En la práctica profesional, esta actualización asegura que los modelos automatizados reflejen fielmente la normativa vigente y eviten inconsistencias frente al SRI.